# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² Mental Health Survey Dataset for Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset is defined via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load the dataset metadata and inspect the top-level information using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset URL (Croissant schema)
CROISSANT_URL = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(CROISSANT_URL)

# Access metadata
meta = dataset.metadata
print(f"{meta.name}: {meta.description}\n\nLicense: {meta.license}\nVersion: {meta.version}\nPublished: {meta.datePublished}")
print(f"Keywords: {meta.keywords}")
print(f"Cite as: {meta.citeAs}")

## 2. Data Overview

Let's list the available **record sets** and their corresponding field and column `@id`s. All referencing uses the schema's internal `@id` values for reproducibility.

In [ ]:
# Inspect all available record sets by ID, along with their fields
record_sets = list(dataset.record_sets())  # Returns objects of type CroissantRecordSet
print(f"Found {len(record_sets)} record sets:\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}, name: {rs.get('name', '')}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Field @ids:")
    for field in fields:
        if isinstance(field, dict):
            print(f"    {field.get('@id')}")
        else:
            print(f"    {field}")
    columns = rs.get('column', [])
    if columns:
        print("  Column @ids:")
        if isinstance(columns, dict):
            columns = [columns]
        for col in columns:
            if isinstance(col, dict):
                print(f"    {col.get('@id')}")
            else:
                print(f"    {col}")
    print()

# Show the first record from the first available record set
if record_sets:
    first_rs_id = record_sets[0]['@id']
    print(f"Sample record from RecordSet {first_rs_id}:")
    for x in dataset.records(record_set=first_rs_id):
        print(x)
        break

## 3. Data Extraction

Now, let's load each available record set into a pandas DataFrame. For further analysis, use record set and field `@id`s for referencing columns.

In [ ]:
# Re-list the record set @ids for reference
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
    else:
        df = pd.DataFrame()
    dataframes[rs_id] = df

if record_set_ids:
    main_id = record_set_ids[0]
    print(f"DataFrame for main record set ({main_id}):")
    print(dataframes[main_id].columns.tolist())
    dataframes[main_id].head()

## 4. Exploratory Data Analysis (EDA)

Let's perform basic processing and EDA using the main survey record set.

- We will select a numeric field (e.g., `phq9_score` or `gad7_score`).
- Filter to rows above a threshold (e.g., score > 10), normalize the score, and group by another attribute such as gender.

**Replace the `phq9_score`, `gender`, etc., below with actual `@id` values present in the main record set's DataFrame.**

In [ ]:
# Detect likely numeric and categorical fields by examining DataFrame
main_df = dataframes[main_id]
print('Likely numeric fields:')
print([col for col in main_df.columns if main_df[col].dtype != object])

# For this dataset, we'll try the following common survey mental health scores
# Replace with actual column names (@id) as needed
if 'phq9_score' in main_df.columns:
    numeric_field = 'phq9_score'
elif 'gad7_score' in main_df.columns:
    numeric_field = 'gad7_score'
elif len(main_df.select_dtypes('number').columns) > 0:
    numeric_field = main_df.select_dtypes('number').columns[0]
else:
    raise Exception('No recognized numeric score field.')

group_field_candidates = ['gender', 'sex', 'village', 'level_of_education']
group_field = None
for candidate in group_field_candidates:
    if candidate in main_df.columns:
        group_field = candidate
        break

# Filter: show respondents with high symptom scores
threshold = 10
filtered_df = main_df[main_df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}: {len(filtered_df)} records")
display_cols = [numeric_field] + ([group_field] if group_field else [])
print(filtered_df[display_cols].head())

# Normalize the score
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a demographic field
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field, f"{numeric_field}_normalized"].mean()
    print(f"Grouped data by {group_field}:")
    print(grouped_df.head())

## 5. Visualization

Let's visualize the distribution of survey scores and explore potential differences by group (e.g., gender or village).


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(main_df[numeric_field], kde=True, bins=20)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

if group_field:
    plt.figure(figsize=(8, 5))
    sns.boxplot(x=group_field, y=numeric_field, data=main_df)
    plt.title(f'{numeric_field} by {group_field}')
    plt.show()

## 6. Conclusion

In this notebook, we've demonstrated how to load, preview, and process the FAIR² Mental Health Survey Dataset from Kilifi using the `mlcroissant` library. By referencing all fields and record sets by their `@id`, you ensure reproducibility across schema updates. This approach enables efficient, standards-compliant machine learning and public health analysis workflows targeting data from Kenya.

> **You can extend this notebook by selecting different record sets, exploring additional fields, and engineering domain-specific visualizations/analyses.**